# 원본 데이터 탐색

이 노트북은 전처리 전에 원본 파일 구조를 점검하고 `wo_id` 기준 참조 무결성을 확인합니다.

- 필수 입력 파일 존재 여부 확인
- 컬럼, 타입, 결측 비율, 고유값 수 확인
- LOT 기준 참조 무결성 점검
- 누락 LOT 분포 분석

In [7]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
INPUT_FILES = {
    "customer_order": DATA_DIR / "MES_customer_order.csv",
    "production_trend": DATA_DIR / "PRODUCTION_TREND.csv",
    "lot_amount": DATA_DIR / "LOT_amount_preprocessed.xlsx",
    "ccm_measure": DATA_DIR / "CCM_measure_preprocessed.xlsx",
}

CSV_ENCODINGS = ("utf-8", "cp949", "euc-kr")


def read_csv_with_fallback(file_path: Path) -> pd.DataFrame:
    latest_exception = None
    for encoding_name in CSV_ENCODINGS:
        try:
            return pd.read_csv(file_path, encoding=encoding_name, low_memory=False)
        except Exception as error:
            latest_exception = error
    raise RuntimeError(f"CSV 파일을 읽을 수 없습니다: {file_path}") from latest_exception

In [8]:
missing_paths = [path for path in INPUT_FILES.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError("입력 파일이 누락되었습니다:\n" + "\n".join(str(path) for path in missing_paths))

frames = {
    "customer_order": read_csv_with_fallback(INPUT_FILES["customer_order"]),
    "production_trend": read_csv_with_fallback(INPUT_FILES["production_trend"]),
    "lot_amount": pd.read_excel(INPUT_FILES["lot_amount"]),
    "ccm_measure": pd.read_excel(INPUT_FILES["ccm_measure"]),
}

for frame_name, data_frame in frames.items():
    print(f"[{frame_name}] 행={len(data_frame):,}, 열={len(data_frame.columns)}")

[customer_order] 행=93,509, 열=21
[production_trend] 행=2,119,545, 열=15
[lot_amount] 행=2,127, 열=6
[ccm_measure] 행=427, 열=4


In [9]:
def profile_frame(frame_name: str, data_frame: pd.DataFrame, max_columns: int = 30) -> None:
    print(f"\n=== {frame_name} ===")
    print("shape:", data_frame.shape)
    print("columns:")
    for column_name in data_frame.columns[:max_columns]:
        print(f"- {column_name}")
    if len(data_frame.columns) > max_columns:
        print(f"... ({len(data_frame.columns) - max_columns}개 컬럼 생략)")

    summary_frame = pd.DataFrame(
        {
            "dtype": data_frame.dtypes.astype(str),
            "null_count": data_frame.isna().sum(),
            "null_ratio": (data_frame.isna().mean() * 100).round(2),
            "nunique": data_frame.nunique(dropna=True),
        }
    )
    display(summary_frame.sort_values(["null_ratio", "nunique"], ascending=[False, False]).head(20))


for frame_name, data_frame in frames.items():
    profile_frame(frame_name, data_frame)


=== customer_order ===
shape: (93509, 21)
columns:
- uid
- order_date
- 제품코드
- 제품명
- 제품계층구조
- 자재유형
- 자재그룹
- 제품등급
- 안전재고
- 현재고량
- 보유율(%)
- 당월총주문량
- 당월총출하량
- 당월주문잔량
- 당월미납잔량
- 과거총주문량
- 과거총출하량
- 과거미납총량
- 과거미납당월출하량
- 과거미납잔량
- 총주문잔량


,dtype,null_count,null_ratio,nunique
과거미납당월출하량,str,13208,14.12,2262
당월미납잔량,str,12064,12.90,3519
과거미납잔량,str,9086,9.72,3105
과거미납총량,str,7663,8.19,1623
당월주문잔량,str,7456,7.97,6302
당월총출하량,str,7007,7.49,6384
총주문잔량,str,5607,6.00,8305
보유율(%),float64,4552,4.87,957
제품계층구조,float64,4277,4.57,46
당월총주문량,str,3870,4.14,7797



=== production_trend ===
shape: (2119545, 15)
columns:
- LOT_NO
- WC_CD
- WC_CNT
- SEQ_NO
- PGM_ID
- RESOURCE_CD
- CR_TEMP
- TRD_TEMP_SP
- TRD_TEMP_PV
- TRD_SPEED1
- TRD_SPEED2
- TRD_SPEED3
- TRD_SPEED4
- INSRT_DT
- PRODUCTION_RESULT_iD


,dtype,null_count,null_ratio,nunique
PRODUCTION_RESULT_iD,str,1817133,85.73,1102
TRD_SPEED1,int64,0,0.00,7365
LOT_NO,str,0,0.00,7003
TRD_SPEED3,int64,0,0.00,6841
TRD_SPEED2,int64,0,0.00,1902
TRD_SPEED4,int64,0,0.00,1588
TRD_TEMP_PV,float64,0,0.00,1417
SEQ_NO,int64,0,0.00,1381
TRD_TEMP_SP,float64,0,0.00,1287
INSRT_DT,str,0,0.00,231



=== lot_amount ===
shape: (2127, 6)
columns:
- LOT번호
- 공정코드
- 투입중량(kg)
- 투입액량(L)
- 단위중량(kg)
- 염색길이(m)


,dtype,null_count,null_ratio,nunique
LOT번호,str,0,0.0,2127
투입중량(kg),float64,0,0.0,1226
염색길이(m),float64,0,0.0,474
단위중량(kg),int64,0,0.0,73
투입액량(L),int64,0,0.0,5
공정코드,str,0,0.0,1



=== ccm_measure ===
shape: (427, 4)
columns:
- LOT번호
- 검사차수
- 작업명
- 염색색차 DE


,dtype,null_count,null_ratio,nunique
LOT번호,str,0,0.0,427
염색색차 DE,float64,0,0.0,427
검사차수,object,0,0.0,3
작업명,str,0,0.0,2


In [10]:
# LOT 기준 참조 무결성 점검
def detect_lot_amount_lot_column(column_names):
    for column_name in column_names:
        upper_name = str(column_name).upper()
        if "LOT" in upper_name:
            return column_name
    return column_names[0]

lot_amount_lot_column_name = detect_lot_amount_lot_column(list(frames["lot_amount"].columns))
print("lot_amount LOT 컬럼:", lot_amount_lot_column_name)

lot_amount_lot_set = set(frames["lot_amount"][lot_amount_lot_column_name].astype(str).str.strip())
production_trend_lot_set = set(frames["production_trend"]["LOT_NO"].astype(str).str.strip())

missing_lot_ids = sorted(production_trend_lot_set - lot_amount_lot_set)
print("production_trend LOT 수:", len(production_trend_lot_set))
print("lot_amount LOT 수:", len(lot_amount_lot_set))
print("누락 LOT 수:", len(missing_lot_ids))

missing_lot_sample_frame = pd.DataFrame({"wo_id": missing_lot_ids[:30]})
missing_lot_sample_frame

lot_amount LOT 컬럼: LOT번호
production_trend LOT 수: 7003
lot_amount LOT 수: 2127
누락 LOT 수: 5106


,wo_id
0,F2004270020
1,F202180008고착
2,F2109150040
3,F2109150041
4,F2110150020
5,F2110150021
6,F2110150022
7,F2110150023
8,F2110150024
9,F2110150025


In [11]:
# 누락 LOT 분포 확인
production_trend_frame = frames["production_trend"].copy()
production_trend_frame["wo_id"] = production_trend_frame["LOT_NO"].astype(str).str.strip()
production_trend_frame["timestamp"] = pd.to_datetime(production_trend_frame["INSRT_DT"], errors="coerce")

missing_log_frame = production_trend_frame[
    production_trend_frame["wo_id"].isin(missing_lot_ids)
].copy()

print("누락 로그 행 수:", len(missing_log_frame))
print("최초 시각:", missing_log_frame["timestamp"].min())
print("최종 시각:", missing_log_frame["timestamp"].max())

display(
    missing_log_frame.groupby("RESOURCE_CD", as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(20)
)

display(
    missing_log_frame.groupby(missing_log_frame["timestamp"].dt.date, as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(20)
)

누락 로그 행 수: 1559657
최초 시각: 2022-01-03 00:00:00
최종 시각: 2022-10-19 00:00:00


,RESOURCE_CD,size
3,FCM04,138758
1,FCM02,123430
24,LCM53,121503
0,FCM01,101725
22,LCM51,101380
16,LCM02,98420
15,LCM01,93635
17,LCM03,93473
2,FCM03,84396
18,LCM04,81065


,timestamp,size
7,2022-01-11,13307
131,2022-06-14,12377
128,2022-06-10,12316
22,2022-01-28,12095
166,2022-07-26,12054
218,2022-10-06,11717
4,2022-01-07,11663
139,2022-06-23,11537
16,2022-01-21,11457
190,2022-08-30,11215


In [12]:
# 후속 분석용 파일 저장
output_debug_dir = DATA_DIR / "debug"
output_debug_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame({"wo_id": missing_lot_ids}).to_csv(
    output_debug_dir / "missing_wo_ids.csv", index=False, encoding="utf-8"
)

missing_log_frame.to_csv(
    output_debug_dir / "missing_wo_logs.csv", index=False, encoding="utf-8"
)

print("저장 완료:")
print("-", output_debug_dir / "missing_wo_ids.csv")
print("-", output_debug_dir / "missing_wo_logs.csv")

저장 완료:
- ..\data\debug\missing_wo_ids.csv
- ..\data\debug\missing_wo_logs.csv
